# 2016〜2026年 再クラスタリング（旧クラスタ0・6を除外）

前回の10クラスタ結果から旧クラスタ0・6の4,810発言を除外し、残る64,138発言について特徴量から作り直してKMeans（k=10固定）を再学習します。旧結果と元Notebookは変更しません。

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'analysis').exists():
    ROOT = ROOT.parent
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.matplotlib'))
Path(os.environ['MPLCONFIGDIR']).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib import font_manager
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(ROOT))
from clustering_2016 import (
    FIXED_K, RANDOM_STATE, apply_manual_labels, cluster_details, compare_k,
    export_results, reduce_dimensions, vectorize,
)

sns.set_theme(style='whitegrid')
mac_font_paths = list(Path('/System/Library/Fonts').glob('*角*W3.ttc'))
if not mac_font_paths:
    raise FileNotFoundError('macOSのヒラギノ角ゴシック W3が見つかりません')
font_manager.fontManager.addfont(str(mac_font_paths[0]))
MAC_JAPANESE_FONT = font_manager.FontProperties(fname=str(mac_font_paths[0])).get_name()
plt.rcParams['font.family'] = MAC_JAPANESE_FONT
plt.rcParams['axes.unicode_minus'] = False
SOURCE = ROOT / 'analysis/clustering_2016_2026/speech_clusters.csv'
OUTPUT_DIR = ROOT / 'analysis/clustering_2016_2026_excluding_0_6'
EXCLUDED_CLUSTERS = {0, 6}
print(f'Matplotlib font: {MAC_JAPANESE_FONT} ({mac_font_paths[0]})')
RANDOM_STATE

## 1. 旧クラスタ0・6を除外

前回結果のクラスタ番号を `previous_cluster` として保存し、除外対象と再学習対象を分離します。

In [1]:
previous = pd.read_csv(SOURCE, encoding='utf-8-sig', parse_dates=['date'])
assert len(previous) == 68948
assert previous['speechID'].is_unique
assert set(previous['cluster']) == set(range(10))
previous = previous.rename(columns={'cluster': 'previous_cluster', 'cluster_label': 'previous_cluster_label'})
exclude_mask = previous['previous_cluster'].isin(EXCLUDED_CLUSTERS)
excluded = previous.loc[exclude_mask].copy()
df = previous.loc[~exclude_mask].copy().reset_index(drop=True)
df['review_flags'] = df['review_flags'].fillna('')
df['year'] = df['year'].astype('int16')
df['month'] = df['date'].dt.to_period('M').astype(str)
assert len(excluded) == 4810
assert len(df) == 64138
assert not df['previous_cluster'].isin(EXCLUDED_CLUSTERS).any()
audit = pd.DataFrame({'category': ['excluded_cluster_0', 'excluded_cluster_6', 'excluded_total', 'recluster_input'], 'count': [(excluded['previous_cluster'] == 0).sum(), (excluded['previous_cluster'] == 6).sum(), len(excluded), len(df)]})
display(audit)
display(pd.crosstab(previous['year'], exclude_mask, colnames=['excluded']).rename(columns={False: 'kept', True: 'excluded'}))

NameError: name 'pd' is not defined

## 2. 特徴量を再作成

除外後の64,138発言だけでTF-IDF語彙とSVDを再学習します。旧モデルのベクトルや重心は再利用しません。

In [ ]:
source_df = df.copy()
vectorizer, tfidf, keep_mask = vectorize(source_df)
zero_vector_records = source_df.loc[~keep_mask].copy()
zero_vector_records['reason'] = 'zero_vector_after_tfidf'
df = source_df.loc[keep_mask].copy().reset_index(drop=True)
svd, reduced = reduce_dimensions(tfidf)
print(f'TF-IDF: {tfidf.shape[0]:,} rows × {tfidf.shape[1]:,} features')
print(f'SVD explained variance: {svd.explained_variance_ratio_.sum():.1%}')
print(f'Zero vectors: {len(zero_vector_records)}')

## 3. KMeans再学習（10クラスタ固定）

In [ ]:
scores, models = compare_k(reduced, candidates=[FIXED_K])
model = models[FIXED_K]
print(f'Fixed k={FIXED_K}')
display(scores.style.format({'silhouette': '{:.4f}', 'inertia': '{:,.1f}'}))

## 4. 新クラスタの解釈

In [ ]:
assignments, summary = cluster_details(df, tfidf, reduced, vectorizer, model)
display(summary.style.format({'ratio': '{:.1%}'}))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_sample = assignments.sample(n=min(10000, len(assignments)), random_state=RANDOM_STATE)
plot_idx = plot_sample.index.to_numpy()
sns.scatterplot(x=reduced[plot_idx, 0], y=reduced[plot_idx, 1], hue=plot_sample['cluster'], palette='tab10', s=10, alpha=.45, legend='full', ax=axes[0])
axes[0].set(title='除外後のSVD先頭2軸（固定標本10,000件）', xlabel='SVD 1', ylabel='SVD 2')
sns.barplot(data=summary, x='cluster', y='count', hue='cluster', palette='tab10', legend=False, ax=axes[1])
axes[1].set_title('新クラスタ件数')
plt.tight_layout();

In [ ]:
for cluster_id in summary['cluster']:
    row = summary.loc[summary['cluster'].eq(cluster_id)].iloc[0]
    members = assignments.loc[assignments['cluster'].eq(cluster_id)]
    representatives = members.nsmallest(5, 'centroid_distance')[['speechID', 'date', 'year', 'speaker', 'speech_cleaned', 'centroid_distance']]
    top_speakers = members['speaker'].value_counts().head(5).rename_axis('speaker').reset_index(name='count')
    print(f"\nNew cluster {cluster_id}: {row['provisional_label']} ({row['count']:,}件)")
    print(f"特徴語: {row['top_terms']}")
    display(top_speakers, representatives)

## 5. 年別クラスタ構成

2026年は途中年です。

In [ ]:
yearly_counts = pd.crosstab(assignments['year'], assignments['cluster'])
yearly_ratio = pd.crosstab(assignments['year'], assignments['cluster'], normalize='index')
display(yearly_counts)
plt.figure(figsize=(14, 6))
sns.heatmap(yearly_ratio.T, cmap='YlGnBu', annot=True, fmt='.0%')
plt.title('除外・再学習後の年別クラスタ構成比（2026年は途中年）')
plt.xlabel('年')
plt.ylabel('新クラスタ')
plt.tight_layout();

## 6. 人手による新ラベル調整

In [ ]:
MANUAL_CLUSTER_LABELS = {
    # 0: '社会保障・医療',
    # 1: '外交・安全保障',
}
assignments, summary = apply_manual_labels(assignments, summary, MANUAL_CLUSTER_LABELS)
display(summary[['cluster', 'cluster_label', 'count', 'ratio', 'top_terms']].style.format({'ratio': '{:.1%}'}))

## 7. 出力と最終検証

除外監査表、新しい発言別クラスタ、要約、年別集計を別ディレクトリへ保存します。

In [ ]:
assert len(assignments) == 64138
assert assignments['speechID'].is_unique
assert assignments['cluster'].notna().all()
assert set(assignments['cluster']) == set(range(10))
assert summary['count'].sum() == len(assignments)
export_results(assignments, summary, OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
excluded.to_csv(OUTPUT_DIR / 'excluded_previous_clusters_0_6.csv', index=False, encoding='utf-8-sig')
yearly_counts.to_csv(OUTPUT_DIR / 'yearly_cluster_counts.csv', encoding='utf-8-sig')
yearly_ratio.to_csv(OUTPUT_DIR / 'yearly_cluster_ratios.csv', encoding='utf-8-sig')
assert len(pd.read_csv(OUTPUT_DIR / 'speech_clusters.csv')) == 64138
assert len(pd.read_csv(OUTPUT_DIR / 'excluded_previous_clusters_0_6.csv')) == 4810
assert len(pd.read_csv(OUTPUT_DIR / 'cluster_summary.csv')) == 10
print(f'出力先: {OUTPUT_DIR}')
print(f'検証完了: 旧クラスタ0・6を4,810件除外、残り{len(assignments):,}件を10クラスタへ再学習')